# Clustering images: benchmark production

Objectif: construire un clustering simple a maintenir en production, sans dependance lourde a un clustering textuel pur.

Decision actuelle: metadata/category buckets d'abord, puis graphe mutual-kNN visuel avec controle semantique leger.

## Architecture cible

```text
images
-> metadata/category buckets
-> nearest neighbors visuels dans chaque bucket
-> mutual kNN graph
-> edge si visuel fort ou visuel + overlap tokens
-> connected components
-> clusters produit
```

Le filtre metadata reste le premier niveau pour le front-end. Le graphe sert a regrouper les images desordonnees.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
while ROOT.name != 'photo-ai-platform' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

BENCHMARK = ROOT / 'ml' / 'experiments' / 'artifact_benchmark.py'
REPORT = ROOT / 'reports' / 'algorithm_tests' / 'latest' / 'summary.json'
BENCHMARK.exists(), REPORT


## Lancer le benchmark complet

Le meme runner evalue texte->image, image->image et clustering. Pour ce notebook on lit surtout la section `clustering` du rapport.

`--cluster-visual-limit 0` exporte toutes les planches des clusters non-singletons. Les singletons sont listes en JSON, car une image seule par cluster n'apporte pas grand-chose visuellement.


In [ ]:
cmd = [
    sys.executable,
    str(BENCHMARK),
    '--top-k', '10',
    '--text-batch-size', '8',
    '--dense-step-visual-limit', '20',
    '--cluster-visual-limit', '0',
    '--category-page-size', '60',
]
subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
summary = json.loads(REPORT.read_text(encoding='utf-8'))
summary['clustering']['metrics']


In [ ]:
import pandas as pd

non_singleton_clusters = summary['clustering']['non_singleton_clusters']
singleton_clusters = summary['clustering']['singleton_clusters']
image_assignments = summary['clustering']['image_assignments']
category_summary = summary['clustering']['category_summary']
cluster_rows = pd.DataFrame([
    {
        'cluster_id': c['cluster_id'],
        'size': c['size'],
        'category': c['category'],
        'coherence': c.get('coherence'),
        'first_caption': c['items'][0]['caption'] if c.get('items') else None,
    }
    for c in non_singleton_clusters
]).sort_values(['size', 'coherence'], ascending=[False, False])
category_rows = pd.DataFrame([
    {'category': category, **values}
    for category, values in category_summary.items()
]).sort_values('images', ascending=False)
print(f"assigned images: {summary['clustering']['metrics']['assigned_images']} / {summary['dataset']['images']}")
print(f"non-singleton clusters: {len(non_singleton_clusters)}")
print(f"singleton clusters: {len(singleton_clusters)}")
display(category_rows)
display(cluster_rows)


## Assignation categorie pour toutes les images

Chaque image a maintenant une ligne dans `image_assignments.json`. Un singleton n'est donc pas une image non traitee: c'est une image assignee a une categorie, mais sans voisin assez fiable pour former un groupe thematique.


In [ ]:
assignments_df = pd.DataFrame(image_assignments)
display(assignments_df.head())
display(assignments_df.groupby(['category', 'assignment_type']).size().unstack(fill_value=0))


In [ ]:
from IPython.display import Image as DisplayImage, display

CATEGORY = category_rows.iloc[0]['category']
CATEGORY_PAGE = 1
category_page = ROOT / 'reports' / 'algorithm_tests' / 'latest' / 'clustering' / 'categories' / CATEGORY / f'{CATEGORY}_page_{CATEGORY_PAGE:03d}.jpg'
print({'category': CATEGORY, 'page': CATEGORY_PAGE, 'path': str(category_page)})
display(DisplayImage(filename=str(category_page)))


In [ ]:
try:
    import matplotlib.pyplot as plt
    buckets = summary['clustering']['metrics']['metadata_buckets']
    plt.figure(figsize=(10, 4))
    plt.bar(list(buckets.keys()), list(buckets.values()))
    plt.xticks(rotation=25, ha='right')
    plt.title('Distribution des buckets metadata/category')
    plt.ylabel('images')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib non installe; afficher buckets directement.')


## Visualiser les clusters

Les planches sont dans `reports/algorithm_tests/latest/clustering/`.

La validation doit verifier trois choses:

- les clusters non-singleton sont coherents visuellement,
- les clusters ne fusionnent pas trop de scenes differentes,
- les singletons restent acceptables pour les images vraiment isolees.


## Inspecter toutes les planches

Les fichiers `cluster_XXXX_size_YYY.jpg` sont generes dans `reports/algorithm_tests/latest/clustering/`. Change `CLUSTER_RANK` pour parcourir les clusters par taille/coherence.


In [ ]:
from IPython.display import Image as DisplayImage, display

CLUSTER_RANK = 0
selected = cluster_rows.iloc[CLUSTER_RANK]
cluster_id = int(selected['cluster_id'])
size = int(selected['size'])
image_path = ROOT / 'reports' / 'algorithm_tests' / 'latest' / 'clustering' / f'cluster_{cluster_id:04d}_size_{size:03d}.jpg'
print(selected.to_dict())
display(DisplayImage(filename=str(image_path)))


In [ ]:
CLUSTER_ID = int(cluster_rows.iloc[0]['cluster_id'])
cluster = next(c for c in non_singleton_clusters if c['cluster_id'] == CLUSTER_ID)
pd.DataFrame(cluster['items'])


## Lecture produit

Cette approche est plus facile a maintenir qu'un clustering textuel lourd. Elle reste explicable: un cluster existe parce que les images sont dans le meme bucket, proches visuellement, et connectees dans un graphe mutual-kNN.

Prochaine amelioration production: ajouter des edges optionnels `same_person`, `same_shooting`, `time_delta`, puis mesurer la precision des clusters sur des albums reels.